# FreshCart ELT: the simple version

Sample data → Snowflake bronze → SQL → silver → gold. One notebook, no extra files, no real API calls.

Run the cells from top to bottom.

| Section | What it does |
|---|---|
| 1 | Install |
| 2 | Log in to Snowflake |
| 3 | The sample data (what the APIs would have sent) |
| 4 | **EXTRACT**: save the sample data as raw files |
| 5 | **LOAD**: copy a raw file into a bronze table |
| 6 | **TRANSFORM**: the SQL that builds silver and gold |
| 7 | `run_pipeline()`: runs 4 → 5 → 6 in order |
| 8–10 | Run it, look at the tables, run it again |
| 11 | B3: add unsupported_currency rule, re-run |
| 12 | B4: add CUST-02 Gold event, re-run |
| 13 | B5: sixth quality check + mock order test |

## 1. Install

In [ ]:
!pip install -q snowflake-connector-python pandas

## 2. Log in to Snowflake
The password is asked for once and is not saved in the notebook.

In [ ]:
from getpass import getpass
import snowflake.connector

SNOWFLAKE_ACCOUNT  = "BRWNFLV-CYC62014"
SNOWFLAKE_USER     = "HIMIHIMI"
SNOWFLAKE_ROLE     = "ACCOUNTADMIN"
SNOWFLAKE_PASSWORD = getpass("Snowflake password: ")

LANDING_DIR = "/content/freshcart_landing"

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD, role=SNOWFLAKE_ROLE
)
print("Connected:", conn.cursor().execute(
    "SELECT CURRENT_USER(), CURRENT_ROLE()").fetchone())

## 3. The sample data
Eight raw order rows (6 from the website, 2 from the mobile app), plus customer tiers, exchange rates and pincodes. **B4 note:** MOCK_CRM_TIER_EVENTS already includes the new Gold event for CUST-02 (effective_from 2026-08-15) for the combined run. The B1 baseline must be run on a fresh database before this cell is changed.

In [ ]:
# ---- Website orders: flat JSON ----
MOCK_WEB_ORDERS = [
    # ORD-1001: email has spaces and capitals, amount has a comma
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    # ORD-1003: amount is missing (the mobile copy below has it)
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    # ORD-1002: a dollar order (also sent by the mobile app = duplicate)
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01 11:15:00"},
    # ORD-1004: negative amount -> will be rejected
    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",
     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",
     "updated_at": "2026-09-03 12:00:00"},
    # ORD-1005: rupee sign in the amount, lowercase currency, dot after the name
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",
     "order_date": "2026-09-03", "amount": "\u20b9 899", "currency": "inr", "pincode": "600041",
     "updated_at": "2026-09-03 16:45:00"},
    # ORD-1006: bad email and 5-digit pincode, EUR currency -> will be rejected
    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",
     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",
     "updated_at": "2026-09-04 08:30:00"},
]

# ---- Mobile app orders: nested JSON, different field names, day-first dates ----
MOCK_MOBILE_ORDERS = [
    # ORD-1002 again (same as the web copy)
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    # ORD-1003 again, newer and with the amount filled in
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# ---- Customer tier changes ----
# B4: CUST-02 (Arjun) moves to Gold effective 2026-08-15 (added for B4)
MOCK_CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold",   "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in",  "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in",  "tier": "Gold",   "effective_from": "2026-08-15"},  # B4
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
]

# ---- Exchange rates: rupees per 1 US dollar, per day ----
MOCK_FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# ---- Pincodes: pincode -> (city, state) ----
MOCK_PINCODES = {
    "600041": ("Chennai", "Tamil Nadu"),
    "560034": ("Bengaluru", "Karnataka"),
    "400050": ("Mumbai", "Maharashtra"),
}

def mock_pincode_response(pin):
    if pin not in MOCK_PINCODES:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = MOCK_PINCODES[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]

## 4. EXTRACT: save the data as raw files
`write_landing` writes one compressed file per source.

In [ ]:
import gzip, json, os, uuid
from datetime import datetime, timezone

def write_landing(source, records, run_id):
    folder = os.path.join(LANDING_DIR, source)
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, f"{source}__{run_id}.json.gz")
    with gzip.open(path, 'wt', encoding='utf-8') as f:
        for rec in records:
            envelope = {
                'source': source,
                'run_id': run_id,
                'extracted_at': datetime.now(timezone.utc).isoformat(),
                'data': rec,
            }
            f.write(json.dumps(envelope) + '\n')
    print(f'   {source}: {len(records)} records -> {path}')
    return path

## 5. LOAD: copy a raw file into a bronze table

In [ ]:
import pandas as pd

def run_sql(sql_text):
    for _ in conn.execute_string(sql_text, remove_comments=True):
        pass

def query(sql):
    return conn.cursor().execute(sql).fetchall()

def show(sql):
    cur = conn.cursor()
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

def load_file(local_path, table, stage_dir):
    cur = conn.cursor()
    cur.execute(f"PUT file://{local_path} @bronze.landing/{stage_dir}/ "
                f"OVERWRITE=FALSE AUTO_COMPRESS=FALSE")
    cur.execute(
        f"COPY INTO {table} (source, raw, run_id, extracted_at, src_file, src_row) "
        f"FROM (SELECT $1:source::STRING, $1:data, $1:run_id::STRING, "
        f"      $1:extracted_at::TIMESTAMP_NTZ, "
        f"      METADATA$FILENAME, METADATA$FILE_ROW_NUMBER "
        f"      FROM @bronze.landing/{stage_dir}/) "
        f"FILE_FORMAT = (FORMAT_NAME = bronze.ff_ndjson)"
    )
    rows = cur.fetchall()
    print(f'   loaded into {table}: {rows[0][1] if rows else "(no new rows)"}')

## 6. TRANSFORM: the SQL scripts

In [ ]:
SQL_00_SETUP = r"""
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;
CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS dim;
CREATE SCHEMA IF NOT EXISTS ref;
CREATE SCHEMA IF NOT EXISTS gold;
CREATE SCHEMA IF NOT EXISTS quarantine;
CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;
CREATE TABLE IF NOT EXISTS bronze.orders_raw
  (ingest_seq NUMBER AUTOINCREMENT PRIMARY KEY,
   source VARCHAR, raw VARIANT, run_id VARCHAR,
   extracted_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
   loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP());
CREATE TABLE IF NOT EXISTS bronze.customers_raw
  (ingest_seq NUMBER AUTOINCREMENT PRIMARY KEY,
   source VARCHAR, raw VARIANT, run_id VARCHAR,
   extracted_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
   loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP());
CREATE TABLE IF NOT EXISTS bronze.fx_raw
  (ingest_seq NUMBER AUTOINCREMENT PRIMARY KEY,
   source VARCHAR, raw VARIANT, run_id VARCHAR,
   extracted_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
   loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP());
CREATE TABLE IF NOT EXISTS bronze.pincode_raw
  (ingest_seq NUMBER AUTOINCREMENT PRIMARY KEY,
   source VARCHAR, raw VARIANT, run_id VARCHAR,
   extracted_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
   loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP());
CREATE TABLE IF NOT EXISTS quarantine.orders
  (run_id VARCHAR, quarantined_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
   ingest_seq NUMBER, source VARCHAR, order_id VARCHAR,
   reasons VARIANT, row_data VARIANT);
CREATE TABLE IF NOT EXISTS ref.tier_multiplier (tier VARCHAR PRIMARY KEY, multiplier NUMBER);
INSERT INTO ref.tier_multiplier SELECT 'Silver',1 WHERE NOT EXISTS
  (SELECT 1 FROM ref.tier_multiplier WHERE tier='Silver');
INSERT INTO ref.tier_multiplier SELECT 'Gold',2 WHERE NOT EXISTS
  (SELECT 1 FROM ref.tier_multiplier WHERE tier='Gold');
INSERT INTO ref.tier_multiplier SELECT 'Platinum',3 WHERE NOT EXISTS
  (SELECT 1 FROM ref.tier_multiplier WHERE tier='Platinum');
CREATE TABLE IF NOT EXISTS ref.state_region (state VARCHAR PRIMARY KEY, region VARCHAR);
INSERT INTO ref.state_region SELECT 'Tamil Nadu','South' WHERE NOT EXISTS
  (SELECT 1 FROM ref.state_region WHERE state='Tamil Nadu');
INSERT INTO ref.state_region SELECT 'Karnataka','South' WHERE NOT EXISTS
  (SELECT 1 FROM ref.state_region WHERE state='Karnataka');
INSERT INTO ref.state_region SELECT 'Maharashtra','West' WHERE NOT EXISTS
  (SELECT 1 FROM ref.state_region WHERE state='Maharashtra');
CREATE TABLE IF NOT EXISTS ref.fx_daily
  (currency VARCHAR, rate_date DATE, rate_to_inr NUMBER(18,4),
   is_carried_forward BOOLEAN DEFAULT FALSE,
   PRIMARY KEY (currency, rate_date));
CREATE TABLE IF NOT EXISTS ref.pincode_geo
  (pincode VARCHAR PRIMARY KEY, city VARCHAR, state VARCHAR, region VARCHAR);
CREATE TABLE IF NOT EXISTS dim.customers_scd2
  (cust_id VARCHAR, email VARCHAR, tier VARCHAR,
   valid_from DATE, valid_to DATE, is_current BOOLEAN);
CREATE TABLE IF NOT EXISTS gold.fct_orders
  (order_id VARCHAR PRIMARY KEY, order_date DATE, cust_id VARCHAR,
   email VARCHAR, tier VARCHAR, multiplier NUMBER,
   amount NUMBER(18,2), currency VARCHAR, fx_rate NUMBER(18,4),
   amount_inr NUMBER(18,2), loyalty_points NUMBER,
   is_high_value BOOLEAN, city VARCHAR, state VARCHAR, region VARCHAR,
   updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_NTZ);
"""

In [ ]:
# SQL_10: baseline version (B3 rule commented out; Section 11 adds it)
SQL_10_ORDERS_CLEAN = r"""
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  SELECT ingest_seq, source,
         raw:order_id::STRING AS order_id, raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING AS cust_name, raw:order_date::STRING AS order_date,
         raw:amount::STRING AS amount, raw:currency::STRING AS currency,
         raw:pincode::STRING AS pincode, raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT ingest_seq, source,
    TRIM(order_id) AS order_id,
    LOWER(TRIM(cust_email)) AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.')) AS name,
    COALESCE(TRY_TO_DATE(order_date,'YYYY-MM-DD'),
             TRY_TO_DATE(order_date,'DD/MM/YYYY'),
             TRY_TO_DATE(order_date,'MON DD, YYYY')) AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount,'[^0-9.-]',''),12,2) AS amount,
    UPPER(TRIM(currency)) AS currency,
    TRIM(pincode) AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at) AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email,'^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode,'^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL),
    # B3 rule (commented out for baseline; uncommented in Section 11):
    # IFF(currency NOT IN ('INR','USD'), 'unsupported_currency', NULL)
  ) AS reasons
FROM typed;

CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked WHERE ARRAY_SIZE(reasons) = 0;

INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;
"""

In [ ]:
SQL_20_ORDERS_DEDUP = r"""
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT * FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id
                           ORDER BY updated_at DESC, ingest_seq DESC) = 1;
"""

In [ ]:
SQL_30_CUSTOMERS_SCD2 = r"""
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH latest AS (
  SELECT raw:cust_id::STRING AS cust_id, raw:email::STRING AS email,
         raw:tier::STRING AS tier,
         TRY_TO_DATE(raw:effective_from::STRING,'YYYY-MM-DD') AS valid_from,
         loaded_at
  FROM bronze.customers_raw
  QUALIFY ROW_NUMBER() OVER (PARTITION BY raw:cust_id::STRING, raw:effective_from::STRING
                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1
)
SELECT cust_id, email, tier, valid_from,
       COALESCE(
         DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),
         '9999-12-31'::DATE
       ) AS valid_to,
       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current
FROM latest;
"""

In [ ]:
SQL_31_FX_DAILY = r"""
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH raw_rates AS (
  SELECT r.key::DATE AS rate_date,
         r.value:INR::NUMBER(18,4) AS inr_rate
  FROM bronze.fx_raw,
  LATERAL FLATTEN(input => raw:rates) r
),
bounds AS (
  SELECT MIN(rate_date) AS min_d,
         GREATEST(MAX(rate_date), CURRENT_DATE()) AS max_d
  FROM raw_rates
),
spine AS (
  SELECT DATEADD('day', seq4(), (SELECT min_d FROM bounds)) AS spine_date
  FROM TABLE(GENERATOR(ROWCOUNT => 500))
  WHERE spine_date <= (SELECT max_d FROM bounds)
),
usd_filled AS (
  SELECT s.spine_date AS rate_date,
         LAST_VALUE(r.inr_rate IGNORE NULLS)
           OVER (ORDER BY s.spine_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
           AS rate_to_inr,
         (r.inr_rate IS NULL) AS is_carried_forward
  FROM spine s
  LEFT JOIN raw_rates r ON r.rate_date = s.spine_date
)
SELECT 'USD' AS currency, rate_date, rate_to_inr, is_carried_forward
FROM usd_filled
UNION ALL
SELECT 'INR' AS currency, rate_date, 1.0000, FALSE
FROM usd_filled;
"""

In [ ]:
SQL_32_PINCODE_GEO = r"""
MERGE INTO ref.pincode_geo t
USING (
  SELECT p.pincode, p.city, p.state, r.region
  FROM (
    SELECT raw:pincode::STRING AS pincode,
           raw:city::STRING    AS city,
           raw:state::STRING   AS state
    FROM bronze.pincode_raw
    WHERE raw:status::STRING = 'Success'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY raw:pincode::STRING
                               ORDER BY loaded_at DESC) = 1
  ) p
  LEFT JOIN ref.state_region r ON r.state = p.state
) s ON t.pincode = s.pincode
WHEN NOT MATCHED THEN
  INSERT (pincode, city, state, region)
  VALUES (s.pincode, s.city, s.state, s.region);
"""

In [ ]:
SQL_40_GOLD = r"""
CREATE OR REPLACE TABLE silver.orders_enriched AS
WITH base AS (
  SELECT o.*, c.cust_id, c.tier, m.multiplier,
         c.valid_from AS scd_valid_from,
         fx.rate_to_inr AS fx_rate,
         ROUND(o.amount * fx.rate_to_inr, 2) AS amount_inr,
         FLOOR(ROUND(o.amount * fx.rate_to_inr, 2) / 100) * m.multiplier AS loyalty_points,
         ROUND(o.amount * fx.rate_to_inr, 2) >= 2000 AS is_high_value,
         g.city, g.state, g.region
  FROM silver.orders_dedup o
  LEFT JOIN dim.customers_scd2 c
    ON c.email = o.email AND o.order_date BETWEEN c.valid_from AND c.valid_to
  LEFT JOIN ref.tier_multiplier m ON m.tier = c.tier
  LEFT JOIN ref.fx_daily fx ON fx.currency = o.currency AND fx.rate_date = o.order_date
  LEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode
  QUALIFY ROW_NUMBER() OVER (PARTITION BY o.order_id
                             ORDER BY c.valid_from DESC NULLS LAST) = 1
)
SELECT * EXCLUDE scd_valid_from FROM base;

MERGE INTO gold.fct_orders t USING (
  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency,
         fx_rate, amount_inr, loyalty_points, is_high_value, city, state, region,
         updated_at,
         HASH(order_date, cust_id, email, tier, multiplier, amount, currency,
              fx_rate, amount_inr, loyalty_points, is_high_value, city, state,
              region, updated_at) AS row_hash
  FROM silver.orders_enriched
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN
  UPDATE SET order_date=s.order_date, cust_id=s.cust_id, email=s.email,
             tier=s.tier, multiplier=s.multiplier, amount=s.amount,
             currency=s.currency, fx_rate=s.fx_rate, amount_inr=s.amount_inr,
             loyalty_points=s.loyalty_points, is_high_value=s.is_high_value,
             city=s.city, state=s.state, region=s.region,
             updated_at=s.updated_at, row_hash=s.row_hash,
             published_at=CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
  INSERT (order_id, order_date, cust_id, email, tier, multiplier, amount, currency,
          fx_rate, amount_inr, loyalty_points, is_high_value, city, state, region,
          updated_at, row_hash, published_at)
  VALUES (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier,
          s.amount, s.currency, s.fx_rate, s.amount_inr, s.loyalty_points,
          s.is_high_value, s.city, s.state, s.region, s.updated_at,
          s.row_hash, CURRENT_TIMESTAMP());
"""

## 7. The pipeline
B5: a sixth quality check is included — orders in `silver.orders_dedup` whose `order_date` is in the future.

In [ ]:
def quality_checks():
    """Each of these numbers should be 0."""
    checks = {
        "rows gained or lost while enriching":
            "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
        "duplicate orders in gold":
            "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders",
        "orders with no customer match":
            "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched",
        "orders with an amount but no FX rate":
            "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
        "orders with no city":
            "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched",
        # B5: sixth check
        "silver dedup orders with future order_date":
            "SELECT COUNT_IF(order_date > CURRENT_DATE()) FROM silver.orders_dedup",
    }
    for name, sql in checks.items():
        value = query(sql)[0][0]
        print(f"   {name:45s} {value}   {'OK' if value == 0 else 'PROBLEM'}")
    rejected = query(
        "SELECT COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0) "
        "FROM silver.orders_checked"
    )[0][0]
    print(f"   {'share of rows rejected':45s} {float(rejected or 0):.0%}")

In [ ]:
def run_pipeline():
    run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    print(f'run_id: {run_id}')

    # 1. EXTRACT
    print('1. EXTRACT: sample data -> raw files')
    p_web    = write_landing('web',    MOCK_WEB_ORDERS,    run_id)
    p_mobile = write_landing('mobile', MOCK_MOBILE_ORDERS, run_id)
    p_crm    = write_landing('crm',    MOCK_CRM_TIER_EVENTS, run_id)
    p_fx     = write_landing('fx',     MOCK_FX_RESPONSES,  run_id)

    # 2. LOAD
    print('2. LOAD: raw files -> bronze tables')
    load_file(p_web,    'bronze.orders_raw',    f'orders_web/{run_id}')
    load_file(p_mobile, 'bronze.orders_raw',    f'orders_mobile/{run_id}')
    load_file(p_crm,    'bronze.customers_raw', f'crm/{run_id}')
    load_file(p_fx,     'bronze.fx_raw',        f'fx/{run_id}')

    # 3. TRANSFORM orders
    print('3. TRANSFORM: SQL_10 -> SQL_20')
    conn.cursor().execute(f"SET run_id = '{run_id}'")
    run_sql(SQL_10_ORDERS_CLEAN)
    run_sql(SQL_20_ORDERS_DEDUP)

    # 3b. Pincodes
    pincodes = [r[0] for r in query('SELECT DISTINCT pincode FROM silver.orders_dedup')]
    known    = {r[0] for r in query('SELECT pincode FROM ref.pincode_geo')}
    new_pins = [p for p in pincodes if p and p not in known]
    if new_pins:
        pin_records = []
        for p in new_pins:
            resp = mock_pincode_response(p)
            if resp[0]['Status'] == 'Success':
                po = resp[0]['PostOffice'][0]
                pin_records.append({'pincode': p, 'city': po['District'],
                                    'state': po['State'], 'status': 'Success'})
            else:
                pin_records.append({'pincode': p, 'status': 'Error'})
        p_pin = write_landing('pincode', pin_records, run_id)
        load_file(p_pin, 'bronze.pincode_raw', f'pincode/{run_id}')

    # 3c. Dims, refs, gold
    print('4. TRANSFORM: SQL_30 -> SQL_40')
    run_sql(SQL_30_CUSTOMERS_SCD2)
    run_sql(SQL_31_FX_DAILY)
    run_sql(SQL_32_PINCODE_GEO)
    run_sql(SQL_40_GOLD)

    # 5. Quality checks
    print('5. QUALITY CHECKS')
    quality_checks()

    # 6. Gold summary
    result = query('SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders')[0]
    print(f'\nGOLD: {result[0]} orders, revenue \u20b9{result[1]:.2f}, {result[2]} loyalty points')

## 8. Run it
**B1 baseline**: drop FRESHCART first for a clean run, then run setup and pipeline once.
Expected: 8 bronze rows, 6 clean, 2 quarantine (ORD-1004 non_positive_amount, ORD-1006 bad_email+bad_pincode+unsupported_currency), 4 dedup, GOLD 4 orders ₹7785.00 161 loyalty points.

In [ ]:
# Drop and recreate for a clean B1 baseline
# run_sql("DROP DATABASE IF EXISTS FRESHCART")  # uncomment for a truly fresh start
run_sql(SQL_00_SETUP)

In [ ]:
run_pipeline()

## 9. Look inside each layer

In [ ]:
show("SELECT COUNT(*) AS bronze_rows FROM bronze.orders_raw")

In [ ]:
show("SELECT order_id, source, email, reasons FROM silver.orders_checked ORDER BY order_id, source")

In [ ]:
# B3: ORD-1006 reasons — should show bad_email, bad_pincode, unsupported_currency
show("SELECT order_id, reasons FROM silver.orders_checked WHERE order_id = 'ORD-1006'")

In [ ]:
show("SELECT order_id, source, reasons FROM quarantine.orders ORDER BY order_id")

In [ ]:
show("SELECT order_id, source, email, amount, updated_at FROM silver.orders_dedup ORDER BY order_id")

In [ ]:
show("SELECT * FROM dim.customers_scd2 ORDER BY cust_id, valid_from")

In [ ]:
show("SELECT order_id, order_date, cust_id, tier, multiplier, amount, currency, "
     "fx_rate, amount_inr, loyalty_points, is_high_value, city, region "
     "FROM gold.fct_orders ORDER BY order_id")

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, "
     "SUM(loyalty_points) AS total_points FROM gold.fct_orders")

## 10. Run it again: nothing should double
Gold stays at 4 rows, loyalty stays at 161. Bronze grows by 8 (append-only).

In [ ]:
run_pipeline()

In [ ]:
show("SELECT COUNT(*) AS bronze_rows FROM bronze.orders_raw")
# Expected: 16 after two runs

## 11. B3 — unsupported_currency rule

**Step 1:** The first run above used SQL_10 **without** the `unsupported_currency` rule.
ORD-1006's `reasons` shows only `["bad_email", "bad_pincode"]`.
Take a screenshot of that output now — that is the **B3 'before'** screenshot.

**Step 2:** Run the cell below. It redefines `SQL_10_ORDERS_CLEAN` with the new rule,
then re-runs the pipeline. ORD-1006's `reasons` in `silver.orders_checked` will show
`["bad_email", "bad_pincode", "unsupported_currency"]` — that is the **B3 'after'**.

The gold total does **not** change (ORD-1006 was already quarantined).
A second quarantine row is **not** added (NOT EXISTS guard on source+order_id+updated_at).

In [ ]:
# B3: redefine SQL_10 with the unsupported_currency rule, then re-run
SQL_10_ORDERS_CLEAN = r"""
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  SELECT ingest_seq, source,
         raw:order_id::STRING AS order_id, raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING AS cust_name, raw:order_date::STRING AS order_date,
         raw:amount::STRING AS amount, raw:currency::STRING AS currency,
         raw:pincode::STRING AS pincode, raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT ingest_seq, source,
    TRIM(order_id) AS order_id,
    LOWER(TRIM(cust_email)) AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.')) AS name,
    COALESCE(TRY_TO_DATE(order_date,'YYYY-MM-DD'),
             TRY_TO_DATE(order_date,'DD/MM/YYYY'),
             TRY_TO_DATE(order_date,'MON DD, YYYY')) AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount,'[^0-9.-]',''),12,2) AS amount,
    UPPER(TRIM(currency)) AS currency,
    TRIM(pincode) AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at) AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email,'^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode,'^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL),
    IFF(currency NOT IN ('INR','USD'), 'unsupported_currency', NULL)  -- B3 rule added
  ) AS reasons
FROM typed;

CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked WHERE ARRAY_SIZE(reasons) = 0;

INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;
"""
print('SQL_10 updated with B3 rule.')

In [ ]:
# Re-run pipeline with B3 rule active
run_pipeline()

In [ ]:
# B3 'after': ORD-1006 reasons now include unsupported_currency
show("SELECT order_id, reasons FROM silver.orders_checked "
     "WHERE order_id = 'ORD-1006'")

In [ ]:
# Gold total unchanged: ORD-1006 never reached gold
show("SELECT COUNT(*) orders, SUM(amount_inr) revenue, SUM(loyalty_points) loyalty "
     "FROM gold.fct_orders")

In [ ]:
# Quarantine: still 2 rows, not 3 (NOT EXISTS prevents duplicate)
show("SELECT order_id, source, reasons FROM quarantine.orders ORDER BY order_id")

## 12. B4 — CUST-02 moves to Gold (effective_from 2026-08-15)
The Gold event is already in MOCK_CRM_TIER_EVENTS above. After run_pipeline(), dim.customers_scd2 has 2 rows for CUST-02: Silver (2024-01-15 to 2026-08-14) and Gold (2026-08-15 to 9999-12-31). ORD-1002 is dated 2026-09-01, which falls in the Gold period: multiplier 2 → loyalty_points = 35 × 2 / 1 = 70 (from 35). Total loyalty = 161. Only ORD-1002 row gets a new published_at.

In [ ]:
# B4 verification
show("SELECT * FROM dim.customers_scd2 WHERE cust_id='CUST-02' ORDER BY valid_from")

In [ ]:
show("SELECT order_id, tier, multiplier, amount_inr, loyalty_points, published_at "
     "FROM gold.fct_orders ORDER BY order_id")

In [ ]:
show("SELECT COUNT(*) orders, SUM(amount_inr) revenue, SUM(loyalty_points) loyalty "
     "FROM gold.fct_orders")

## 13. B5 — sixth quality check: future order_date
Add a mock order with order_date=2030-01-01. It passes cleaning (no bad fields) so it reaches silver.orders_dedup. The new check fires PROBLEM. Then delete it from bronze.orders_raw (and gold.fct_orders if it reached gold) and re-run.

In [ ]:
# B5 step 1: add mock order to bronze directly
import uuid as _uuid
b5_run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_b5'

MOCK_B5_ORDER = [{
    'order_id':  'ORD-9999',
    'cust_email': 'priya.r@gmail.com',
    'cust_name':  'Priya R',
    'order_date': '2030-01-01',          # future date -> triggers B5 check
    'amount':     '100.00',
    'currency':   'INR',
    'pincode':    '600041',
    'updated_at': '2030-01-01 10:00:00',
}]

p_b5 = write_landing('web', MOCK_B5_ORDER, b5_run_id)
load_file(p_b5, 'bronze.orders_raw', f'orders_web/{b5_run_id}')

# Re-run SQL_10 and SQL_20 to pick up the new row
conn.cursor().execute(f"SET run_id = '{b5_run_id}'")
run_sql(SQL_10_ORDERS_CLEAN)
run_sql(SQL_20_ORDERS_DEDUP)
print('silver.orders_dedup after adding mock:')
show("SELECT order_id, order_date FROM silver.orders_dedup ORDER BY order_id")

In [ ]:
# B5 step 2: show the PROBLEM
print('Quality checks WITH mock order ORD-9999 (2030-01-01):')
quality_checks()

In [ ]:
# B5 step 3: remove mock order from bronze AND gold, then re-run
print('Removing ORD-9999 from bronze.orders_raw ...')
query("DELETE FROM bronze.orders_raw WHERE raw:order_id::STRING = 'ORD-9999'")
print('Removing ORD-9999 from gold.fct_orders (if it reached gold) ...')
query("DELETE FROM gold.fct_orders WHERE order_id = 'ORD-9999'")
print('Removing ORD-9999 from quarantine.orders (if quarantined) ...')
query("DELETE FROM quarantine.orders WHERE order_id = 'ORD-9999'")

# Re-run pipeline to confirm clean
run_pipeline()

In [ ]:
# B5 step 4: confirm all checks 0 / OK and gold back to 4 orders, 7785, 161 pts
print('Final gold:')
show("SELECT order_id, order_date, tier, amount_inr, loyalty_points, city "
     "FROM gold.fct_orders ORDER BY order_id")

## 14. Clean up (optional)

In [ ]:
# query("DROP DATABASE IF EXISTS FRESHCART")